# Crawl & Build Dataset — Phát hiện lỗi chính tả tiếng Anh

**Đề tài**: Phát hiện lỗi chính tả trong bài luận tiếng Anh của học viên  
**Môn**: Cơ sở dữ liệu nâng cao  
**Output**: bảng `lang_8 (id, source, target)` trong PostgreSQL — gốc cho fine-tune T5

## Pipeline
1. **Lang-8 Kaggle** — corpus chính (~1M cặp)
2. **JFLEG** (HuggingFace) — bài luận đã được 4 người sửa độc lập
3. **W&I+LOCNESS** (HuggingFace `wi_locness`) — bài luận học viên ESL
4. **Synthetic noise injection** — sinh lỗi nhân tạo từ Brown corpus
5. **GitHub Typo Corpus** — typo thật từ commit messages
6. **Reddit r/EnglishLearning** — cặp before/after correction

Sau đó **Clean → Dedup → Filter → Stats → Push DB / Save CSV**.

Chạy: Runtime → Run all (Colab Pro với GPU không bắt buộc cho notebook này).

## 0. Cài thư viện

In [ ]:
!pip install -q pandas datasets requests kaggle psycopg2-binary nltk tqdm matplotlib pyngrok

In [ ]:
import os, re, random, json, time, uuid
from pathlib import Path
import pandas as pd
import requests
from tqdm.auto import tqdm
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('brown', quiet=True)

random.seed(42)
OUT_DIR = Path('./crawled_data'); OUT_DIR.mkdir(exist_ok=True)
TARGET_SIZE = 50_000  # số cặp mong muốn của dataset cuối cùng

## 1. Nguồn 1 — Lang-8 Corpus (Kaggle)

**Cách lấy Kaggle API token**:
1. Vào https://www.kaggle.com/settings → Create New Token → tải `kaggle.json`
2. Upload `kaggle.json` vào Colab khi cell hỏi

In [ ]:
from google.colab import files
import os, shutil

if not Path('~/.kaggle/kaggle.json').expanduser().exists():
    print('Upload kaggle.json:')
    up = files.upload()
    Path('~/.kaggle').expanduser().mkdir(exist_ok=True)
    shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Dataset Lang-8 phổ biến trên Kaggle
!kaggle datasets download -d shanyl/lang8 -p {OUT_DIR} --unzip || \
 kaggle datasets download -d zonghaoyang/lang-8-corpus -p {OUT_DIR} --unzip || \
 echo 'Nếu lỗi: vào kaggle.com tìm dataset Lang-8 và sửa tên dataset trên đây'

In [ ]:
def load_lang8():
    """Tự dò file Lang-8 trong OUT_DIR — chấp nhận nhiều format."""
    candidates = list(OUT_DIR.glob('*.csv')) + list(OUT_DIR.glob('*.tsv')) + list(OUT_DIR.glob('*.txt'))
    for f in candidates:
        try:
            df = pd.read_csv(f, sep=None, engine='python', on_bad_lines='skip')
            cols = [c.lower() for c in df.columns]
            # Chuẩn hoá tên cột
            if 'source' in cols and 'target' in cols:
                df.columns = cols
                return df[['source', 'target']]
            if 'incorrect' in cols and 'correct' in cols:
                df.columns = cols
                return df.rename(columns={'incorrect':'source','correct':'target'})[['source','target']]
        except Exception:
            pass
    return pd.DataFrame(columns=['source','target'])

lang8 = load_lang8()
print(f'Lang-8: {len(lang8)} cặp')
lang8.head(3)

## 2. Nguồn 2 — JFLEG (HuggingFace)
Bài luận có 4 phiên bản sửa độc lập → rất tốt cho test set.

In [ ]:
from datasets import load_dataset

try:
    jfleg = load_dataset('jfleg', split='validation+test')
    rows = []
    for ex in jfleg:
        for corr in ex['corrections']:
            if corr.strip() and corr.strip() != ex['sentence'].strip():
                rows.append((ex['sentence'].strip(), corr.strip()))
    jfleg_df = pd.DataFrame(rows, columns=['source', 'target'])
    print(f'JFLEG: {len(jfleg_df)} cặp')
except Exception as e:
    print('JFLEG fail:', e)
    jfleg_df = pd.DataFrame(columns=['source','target'])

## 3. Nguồn 3 — W&I+LOCNESS (HuggingFace)
Bài luận học viên ESL các trình độ A→C, có annotation.

In [ ]:
try:
    # Một số mirror trên HF: 'wi_locness', 'GEM/wiki_lingua' không phải, 'lemon42-ai/grammar_correction'
    wi = load_dataset('lemon42-ai/grammar_correction', split='train')
    rows = [(r['input'].strip(), r['output'].strip()) for r in wi
            if r['input'].strip() and r['output'].strip() and r['input'] != r['output']]
    wi_df = pd.DataFrame(rows, columns=['source','target'])
    print(f'W&I-like: {len(wi_df)} cặp')
except Exception as e:
    print('W&I fail (bỏ qua):', e)
    wi_df = pd.DataFrame(columns=['source','target'])

## 4. Nguồn 4 — Synthetic noise injection (Brown corpus)
Sinh lỗi đánh máy & homophone từ câu sạch. Hữu ích để **cân bằng** dataset (Lang-8 lệch nặng về lỗi grammar; spell-checking cần thêm data lỗi typo).

In [ ]:
from nltk.corpus import brown

KEYBOARD = {
    'a':'qwsz','b':'vghn','c':'xdfv','d':'sefcx','e':'wsdr','f':'drtgvc','g':'ftyhbv',
    'h':'gyujnb','i':'ujko','j':'huiknm','k':'jiolm','l':'kop','m':'njk','n':'bhjm',
    'o':'iklp','p':'ol','q':'wa','r':'edft','s':'awedxz','t':'rfgy','u':'yhji','v':'cfgb',
    'w':'qase','x':'zsdc','y':'tghu','z':'asx'
}
HOMOPHONES = [
    ('there','their'),('your',"you're"),('its',"it's"),('then','than'),
    ('to','too'),('affect','effect'),('lose','loose'),('accept','except'),
    ('weather','whether'),('which','witch'),('write','right'),('know','no')
]
COMMON_SWAPS = [
    ('information','informations'),('advice','advices'),('went','goed'),
    ('children','childrens'),('does','do'),('was','were')
]

def add_typo(word):
    if len(word) < 3 or not word.isalpha():
        return word
    op = random.choice(['neighbor','delete','duplicate','swap'])
    i = random.randint(0, len(word)-1)
    c = word[i].lower()
    if op == 'neighbor' and c in KEYBOARD:
        return word[:i] + random.choice(KEYBOARD[c]) + word[i+1:]
    if op == 'delete':
        return word[:i] + word[i+1:]
    if op == 'duplicate':
        return word[:i] + word[i] + word[i:]
    if op == 'swap' and i < len(word)-1:
        return word[:i] + word[i+1] + word[i] + word[i+2:]
    return word

def corrupt(sent, rate=0.15):
    words = sent.split()
    out = [add_typo(w) if random.random() < rate else w for w in words]
    text = ' '.join(out)
    if random.random() < 0.3:
        for correct, wrong in HOMOPHONES:
            if re.search(rf'\\b{correct}\\b', text) and random.random() < 0.5:
                text = re.sub(rf'\\b{correct}\\b', wrong, text, count=1)
                break
    if random.random() < 0.2:
        for correct, wrong in COMMON_SWAPS:
            if correct in text and random.random() < 0.5:
                text = text.replace(correct, wrong, 1)
                break
    return text

clean_sents = [' '.join(s) for s in brown.sents() if 5 <= len(s) <= 30]
random.shuffle(clean_sents)
clean_sents = clean_sents[:15000]
synthetic = [(corrupt(s), s) for s in clean_sents]
synthetic = [(s, t) for s, t in synthetic if s != t]
synth_df = pd.DataFrame(synthetic, columns=['source','target'])
print(f'Synthetic: {len(synth_df)} cặp')
synth_df.head(3)

## 5. Nguồn 5 — GitHub Typo Corpus
Typo thật từ commit messages, parse trực tiếp từ release JSONL.

In [ ]:
import gzip, urllib.request

TYPO_URL = 'https://github.com/mhagiwara/github-typo-corpus/releases/download/v1.0.0/github-typo-corpus.v1.0.0.jsonl.gz'
typo_path = OUT_DIR / 'typo_corpus.jsonl.gz'
if not typo_path.exists():
    print('Đang tải GitHub Typo Corpus (~50MB)...')
    try:
        urllib.request.urlretrieve(TYPO_URL, typo_path)
    except Exception as e:
        print('Tải fail:', e)

rows = []
if typo_path.exists():
    with gzip.open(typo_path, 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 200_000: break  # giới hạn để tiết kiệm RAM
            try:
                obj = json.loads(line)
                for edit in obj.get('edits', []):
                    if edit.get('is_typo') and edit.get('src') and edit.get('tgt'):
                        src, tgt = edit['src']['text'], edit['tgt']['text']
                        if 4 <= len(src.split()) <= 40 and src != tgt:
                            rows.append((src, tgt))
            except Exception:
                continue
typo_df = pd.DataFrame(rows, columns=['source','target']).drop_duplicates()
print(f'GitHub Typo: {len(typo_df)} cặp')

## 6. Nguồn 6 — Reddit r/EnglishLearning
Crawl public JSON, không cần OAuth. Pattern phổ biến trong comments: "Original: ... / Correction: ...".

In [ ]:
def crawl_reddit(subreddits=('EnglishLearning','grammar','writing'), per_sub=300):
    headers = {'User-Agent': 'csdl-nang-cao-study/1.0'}
    pairs = []
    pattern = re.compile(
        r'(?:original|wrong|incorrect)[:\s]+(.+?)(?:\n+|\.\s).*?(?:correct(?:ion|ed)?|fixed|right)[:\s]+(.+?)(?:\n|$)',
        re.IGNORECASE | re.DOTALL
    )
    for sub in subreddits:
        after = ''
        for _ in range(per_sub // 25):
            url = f'https://www.reddit.com/r/{sub}/new.json'
            try:
                r = requests.get(url, params={'limit':25,'after':after}, headers=headers, timeout=10)
                if r.status_code != 200: break
                data = r.json().get('data', {})
                after = data.get('after','')
                for child in data.get('children', []):
                    body = (child['data'].get('selftext','') + '\n' + child['data'].get('title',''))
                    for m in pattern.finditer(body):
                        src, tgt = m.group(1).strip(), m.group(2).strip()
                        if 4 < len(src.split()) < 40 and src.lower() != tgt.lower():
                            pairs.append((src, tgt))
                if not after: break
                time.sleep(1.5)  # respect rate limit
            except Exception:
                break
    return pd.DataFrame(pairs, columns=['source','target']).drop_duplicates()

reddit_df = crawl_reddit()
print(f'Reddit: {len(reddit_df)} cặp')

## 7. Hợp nhất + Làm sạch + Dedup

Quy tắc filter:
- 3 ≤ độ dài câu ≤ 50 từ
- src ≠ tgt (không có lỗi)
- Bỏ ký tự non-ASCII (emoji, ngôn ngữ khác)
- Bỏ URL, code block
- Bỏ duplicate

In [ ]:
all_dfs = {
    'lang8':     lang8,
    'jfleg':     jfleg_df,
    'wi':        wi_df,
    'synthetic': synth_df,
    'github':    typo_df,
    'reddit':    reddit_df,
}
for name, d in all_dfs.items():
    d['origin'] = name

raw = pd.concat(all_dfs.values(), ignore_index=True)
print('Trước clean:', len(raw))

def is_valid(s, t):
    if not isinstance(s,str) or not isinstance(t,str): return False
    s, t = s.strip(), t.strip()
    if not s or not t: return False
    if s.lower() == t.lower(): return False
    if not (3 <= len(s.split()) <= 50): return False
    if not (3 <= len(t.split()) <= 50): return False
    if re.search(r'[^\x00-\x7F]', s) or re.search(r'[^\x00-\x7F]', t): return False
    if 'http' in s.lower() or 'http' in t.lower(): return False
    if '```' in s or '```' in t: return False
    return True

raw['source'] = raw['source'].astype(str).str.replace(r'\\s+',' ',regex=True).str.strip()
raw['target'] = raw['target'].astype(str).str.replace(r'\\s+',' ',regex=True).str.strip()
mask = raw.apply(lambda r: is_valid(r['source'], r['target']), axis=1)
clean = raw[mask].drop_duplicates(subset=['source','target']).reset_index(drop=True)
print('Sau clean:', len(clean))

# Cân bằng: giới hạn mỗi nguồn không quá 40% dataset
max_per_source = int(TARGET_SIZE * 0.4)
balanced = (clean.groupby('origin', group_keys=False)
                 .apply(lambda g: g.sample(min(len(g), max_per_source), random_state=42)))
balanced = balanced.sample(frac=1, random_state=42).reset_index(drop=True)
if len(balanced) > TARGET_SIZE:
    balanced = balanced.head(TARGET_SIZE)
print('Final balanced:', len(balanced))
balanced['origin'].value_counts()

## 8. Thống kê + biểu đồ (cho báo cáo)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
balanced['origin'].value_counts().plot.bar(ax=axes[0], title='Số cặp theo nguồn')
balanced['source'].str.split().str.len().hist(bins=30, ax=axes[1])
axes[1].set_title('Độ dài câu source (từ)')
edit_dist = (balanced['source'].str.len() - balanced['target'].str.len()).abs()
edit_dist.hist(bins=30, ax=axes[2])
axes[2].set_title('|len(src)-len(tgt)|')
plt.tight_layout(); plt.savefig(OUT_DIR/'stats.png', dpi=150)
plt.show()

print('\nMẫu ngẫu nhiên:')
for _, r in balanced.sample(5, random_state=1).iterrows():
    print(f"[{r.origin}]")
    print('  src:', r.source)
    print('  tgt:', r.target)

## 9. Lưu CSV + Tải về máy

In [ ]:
out_csv = OUT_DIR / 'data.csv'
balanced[['source','target']].to_csv(out_csv, index=False, encoding='utf-8')
print('Saved:', out_csv, balanced.shape)

from google.colab import files
files.download(str(out_csv))

## 10. (Tuỳ chọn) Đẩy thẳng vào PostgreSQL bảng `lang_8`

Vì DB của bạn chạy ở localhost Windows, cần expose qua **ngrok TCP tunnel**. Trên máy Windows mở terminal riêng:
```
ngrok tcp 5432
```
→ ngrok cho ra `tcp://X.tcp.ngrok.io:YYYY`. Paste host & port vào cell dưới.

In [ ]:
import psycopg2
from psycopg2.extras import execute_values

DB_CONFIG = {
    'host':     '0.tcp.ngrok.io',   # đổi thành host ngrok của bạn
    'port':     12345,              # đổi thành port ngrok
    'dbname':   'postgres',
    'user':     'postgres',
    'password': 'root',
}
SCHEMA = 'public'  # hoặc 'db_assignment'

def push_to_db(df, batch=1000):
    conn = psycopg2.connect(**DB_CONFIG)
    try:
        with conn:
            with conn.cursor() as cur:
                cur.execute(f'TRUNCATE TABLE {SCHEMA}.lang_8 RESTART IDENTITY')
                rows = list(zip(df['source'].tolist(), df['target'].tolist()))
                for i in tqdm(range(0, len(rows), batch)):
                    execute_values(cur,
                        f'INSERT INTO {SCHEMA}.lang_8 (source, target) VALUES %s',
                        rows[i:i+batch])
        print(f'Đã đẩy {len(df)} cặp vào {SCHEMA}.lang_8')
    finally:
        conn.close()

# Bỏ comment khi đã setup ngrok:
# push_to_db(balanced)

## 11. (Tuỳ chọn) Push lên GitHub repo `data_group7`
Để notebook training của bạn đọc được qua URL `https://raw.githubusercontent.com/.../data.csv`.

In [ ]:
# Cần GitHub Personal Access Token (Settings → Developer settings → Tokens)
GITHUB_TOKEN = ''  # paste token vào đây
GITHUB_USER  = 'TangNgocPhung'
GITHUB_REPO  = 'data_group7'

if GITHUB_TOKEN:
    !git config --global user.email "colab@study.local"
    !git config --global user.name "{GITHUB_USER}"
    !rm -rf /content/{GITHUB_REPO}
    !git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{GITHUB_REPO}.git /content/{GITHUB_REPO}
    !cp {out_csv} /content/{GITHUB_REPO}/data.csv
    !cd /content/{GITHUB_REPO} && git add data.csv && git commit -m 'update dataset' && git push

---
## Tóm tắt cải tiến so với pipeline cũ
| Phần | Cũ | Mới |
|---|---|---|
| Số nguồn data | 3 (Kaggle / crawl không rõ / chatbot) | 6 nguồn đa dạng |
| Cân bằng | Không có | Mỗi nguồn ≤ 40% dataset |
| Filter chất lượng | Không có | 7 quy tắc validate |
| Dedup | Không có | drop_duplicates trên (src,tgt) |
| Thống kê cho báo cáo | Không có | 3 biểu đồ + sample 5 cặp |
| Đẩy DB | Manual | Auto qua ngrok + execute_values |
| Push GitHub | Manual | Auto bằng PAT |
